# Zucchini CNN Regression — CC/NOCC × Color-Space Ablation Study (5-Fold CV)

Built on top of `zucchini_cnn_regression_128.ipynb` — same `RegressionCNN` architecture
(6 conv blocks, 128×128 input, single regression head, MSE loss), reused **unmodified**
so results are directly comparable to that baseline.

**Two ablation axes, fully crossed:**

| Axis | Values |
|---|---|
| Color-correction filter | `cc`, `nocc` |
| Color space | `RGB`, `HSV`, `LAB`, `XYZ` |

→ 8 combinations, each evaluated with **stratified 5-fold cross-validation** (score = stratification label).

**Data layout expected** (this is the part that changed vs. the original notebook — verify §2 output
before doing anything else):

```
OUTPUT_ROOT/
├── cc/
│   └── CSVS01/
│       ├── Score=01/*.tif
│       ├── Score=02/*.tif
│       ...
│       └── Score=09/*.tif
└── nocc/
    └── CSVS01/
        ├── Score=01/*.tif
        ...
```

No more filename-based `+cc+` / `+nocc+` filtering — the split is now the **top-level folder**,
and classes are **`Score=XX` subfolders** instead of bare numbers. §2 below discovers these folders
and prints per-score image counts for both conditions — **run it and check the counts before
launching the full ablation grid.**

⚠️ **Compute note:** 2 filters × 4 color spaces × 5 folds = **40 independent trainings**. With the
project's standard settings (400 epochs, early-stop patience 30) this can run for many hours.
Set `QUICK_MODE = True` (default below) for a fast smoke test of the whole pipeline first — it
drops epochs/patience/folds so you can confirm the loader, color-space conversion, and training
loop all work end-to-end before committing to the full overnight run.

This notebook is meant to run **on your machine** (it reads from the `F:\...` path directly) — the
sandbox that generated it has no access to that dataset, so nothing here has been executed.

## 1. Imports & Environment Setup

Same kernel-crash mitigation as `zucchini_cnn_regression_128.ipynb` (`KMP_DUPLICATE_LIB_OK`, `MKL_THREADING_LAYER=GNU`, thread caps, deferred/guarded CUDA check). If you hit the OpenMP crash, **Restart Kernel** (not just re-run this cell) before proceeding.

In [ ]:
import os

# ── Kernel-crash mitigation: MUST run before importing numpy/torch/scipy/skimage ──
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_THREADING_LAYER"]  = "GNU"
os.environ["OMP_NUM_THREADS"]      = "4"
os.environ["MKL_NUM_THREADS"]      = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"]  = "4"

import re
import json
import random
import warnings
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import tqdm
from PIL import Image
from skimage import io, color as skcolor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
%matplotlib inline
mpl.rcParams["figure.figsize"] = 14, 7
plt.rcParams["font.size"] = 10

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.set_num_threads(4)

try:
    _cuda_ok = torch.cuda.is_available()
except Exception as e:
    print(f"⚠ torch.cuda.is_available() raised {type(e).__name__}: {e} -- falling back to CPU.")
    _cuda_ok = False

device = torch.device("cuda" if _cuda_ok else "cpu")
print(f"Using device: {device}")
if _cuda_ok:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn present | scikit-image present")

## 2. Configuration

Edit `OUTPUT_ROOT` / `CSVS_FOLDER` if needed. Everything else follows the project's established ablation-study conventions (`IMG_SIZE=128`, `BATCH_SIZE=32`, `SEED=42`, `K_FOLDS=5`, `TARGET_NOISE_STD=0.15`, early-stop patience 30, LR-scheduler patience 8).

In [ ]:
# ── Ablation axes ──
FILTER_CONDITIONS = ["cc", "nocc"]                  # top-level folder = color-correction status
COLOR_SPACES       = ["RGB", "HSV", "LAB", "XYZ"]    # applied on top of the loaded 16-bit TIFF

# ── Data location ──
OUTPUT_ROOT = r"F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT"
CSVS_FOLDER = "CSVS01"          # device/batch folder nested under cc/ and nocc/
# Regex for "Score=01", "Score=1", "score_09", etc. -- zero-padding agnostic, case-insensitive.
SCORE_FOLDER_REGEX = re.compile(r"score[=_-]?0*(\d+)$", re.IGNORECASE)

# ── Image / training params (kept consistent with the rest of the Veggie Vision ablation notebooks) ──
IMG_SIZE           = 128
BATCH_SIZE         = 32
LEARNING_RATE      = 1e-3
LR_SCHED_PATIENCE  = 8
TARGET_NOISE_STD   = 0.15   # Gaussian noise added to training targets only (regularization)
K_FOLDS            = 5

# Full-scale settings (used as-is unless QUICK_MODE below overrides the *ablation grid* only --
# the final winning-combo deep dive in §10 always trains at full scale, see FINAL_* below).
EPOCHS               = 400
EARLY_STOP_PATIENCE  = 30

# Always-full-scale settings for the §10 deep dive on the winning combination.
FINAL_EPOCHS              = 400
FINAL_EARLY_STOP_PATIENCE = 30

# ── QUICK_MODE: smoke-test the whole pipeline (loader → color-space conversion → CV loop → plots)
#    on a handful of epochs/folds before committing to the full 40-run grid. Flip to False for the
#    real run. Does NOT affect the §10 final deep-dive, which always uses FINAL_* above. ──
QUICK_MODE = False
ABLATION_EPOCHS              = EPOCHS
ABLATION_EARLY_STOP_PATIENCE = EARLY_STOP_PATIENCE
ABLATION_K_FOLDS             = K_FOLDS
if QUICK_MODE:
    ABLATION_EPOCHS, ABLATION_EARLY_STOP_PATIENCE, ABLATION_K_FOLDS = 15, 5, 2
    print("⚠ QUICK_MODE is ON -- these are smoke-test settings (15 epochs, patience 5, 2 folds), "
          "NOT the real ablation. Set QUICK_MODE = False once the pipeline checks out.")
else:
    print(f"Full run: {EPOCHS} epochs, patience {EARLY_STOP_PATIENCE}, {K_FOLDS}-fold CV "
          f"× {len(FILTER_CONDITIONS)} filters × {len(COLOR_SPACES)} color spaces = "
          f"{len(FILTER_CONDITIONS) * len(COLOR_SPACES) * K_FOLDS} trainings total.")

if not os.path.exists(OUTPUT_ROOT):
    print(f"⚠ WARNING: OUTPUT_ROOT does not exist on this machine:\n  {OUTPUT_ROOT}")
    print("  Double-check the path before continuing -- discovery below will find 0 images otherwise.")

## 3. Dataset Discovery & Verification

**Run this before anything else.** It walks `OUTPUT_ROOT/<cc|nocc>/CSVS01/Score=XX/*.tif`, parses the score from each folder name with `SCORE_FOLDER_REGEX`, and prints a per-score image count for both filter conditions. If a count looks wrong (0 images, missing scores, folders not matching the regex), fix `OUTPUT_ROOT`, `CSVS_FOLDER`, or `SCORE_FOLDER_REGEX` here rather than downstream.

In [ ]:
def discover_score_folders(csvs_path, pattern=SCORE_FOLDER_REGEX):
    '''Find Score=XX subfolders under a CSVS folder and parse their numeric score.
    Returns a list of (folder_path, score:int) sorted by score.'''
    found = []
    if not os.path.isdir(csvs_path):
        return found
    for entry in sorted(os.listdir(csvs_path)):
        full = os.path.join(csvs_path, entry)
        if not os.path.isdir(full):
            continue
        m = pattern.search(entry)
        if m:
            found.append((full, int(m.group(1))))
        else:
            print(f"    (ignoring non-matching subfolder: '{entry}')")
    return sorted(found, key=lambda x: x[1])


def find_images_for_condition(output_root, filter_condition, csvs_folder, verbose=True):
    '''Walk OUTPUT_ROOT/<filter_condition>/<csvs_folder>/Score=XX/*.tif(f).
    Returns list of (image_path, score:float) tuples.'''
    csvs_path = os.path.join(output_root, filter_condition, csvs_folder)
    data = []
    score_folders = discover_score_folders(csvs_path)
    if not score_folders:
        if verbose:
            print(f"  ⚠ No 'Score=XX' folders found under {csvs_path}")
        return data
    for folder_path, score in score_folders:
        tifs = glob(os.path.join(folder_path, "*.tif")) + glob(os.path.join(folder_path, "*.tiff"))
        data.extend((p, float(score)) for p in tifs)
        if verbose:
            print(f"    Score={score:02d}: {len(tifs):4d} images  ({os.path.basename(folder_path)})")
    return data


print("=" * 80)
print("DATASET DISCOVERY")
print("=" * 80)
discovered = {}
for cond in FILTER_CONDITIONS:
    csvs_path = os.path.join(OUTPUT_ROOT, cond, CSVS_FOLDER)
    print(f"\n[{cond.upper()}]  {csvs_path}")
    data = find_images_for_condition(OUTPUT_ROOT, cond, CSVS_FOLDER)
    discovered[cond] = data
    print(f"  → Total: {len(data)} images")

print("\n" + "=" * 80)
for cond, data in discovered.items():
    if len(data) == 0:
        print(f"⚠ WARNING: 0 images found for '{cond}'. This condition will be SKIPPED in the "
              f"ablation loop below. Fix the path/regex before running the full grid.")
if all(len(d) == 0 for d in discovered.values()):
    raise RuntimeError("No images found for ANY filter condition -- check OUTPUT_ROOT and CSVS_FOLDER.")

In [ ]:
# Visual sanity check: score distribution, side by side for cc vs nocc
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
all_scores = sorted({int(s) for d in discovered.values() for _, s in d})
x = np.arange(len(all_scores))

for i, cond in enumerate(FILTER_CONDITIONS):
    scores = [s for _, s in discovered.get(cond, [])]
    counts = [scores.count(float(sc)) for sc in all_scores]
    ax.bar(x + (i - 0.5) * width, counts, width, label=cond, edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels([f"Score={s:02d}" for s in all_scores])
ax.set_xlabel("Score class", fontsize=12, fontweight="bold")
ax.set_ylabel("Number of images", fontsize=12, fontweight="bold")
ax.set_title("Class distribution by filter condition", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. Color-Space Conversion

Each image is loaded once as normalized RGB (`[0,1]`, from the 16-bit TIFF) and then converted into the target color space. Every space is rescaled to its known theoretical range and mapped to `[-1, 1]`, matching the `Normalize(mean=0.5, std=0.5)` convention the base RGB notebook already uses -- so the network sees a comparably-scaled input regardless of which color space is being tested.

In [ ]:
COLOR_SPACE_RANGES = {
    # (low, high) per channel, in the color space's native scale
    "RGB": (np.array([0.0, 0.0, 0.0]),     np.array([1.0, 1.0, 1.0])),
    "HSV": (np.array([0.0, 0.0, 0.0]),     np.array([1.0, 1.0, 1.0])),
    "LAB": (np.array([0.0, -128.0, -128.0]), np.array([100.0, 127.0, 127.0])),
    "XYZ": (np.array([0.0, 0.0, 0.0]),     np.array([0.95, 1.0, 1.09])),
}

def convert_and_normalize(rgb01, color_space):
    '''rgb01: float32 (H,W,3) in [0,1], sRGB. Returns float32 (H,W,3) in ~[-1,1]
    in the requested color space.'''
    if color_space == "RGB":
        out = rgb01
    elif color_space == "HSV":
        out = skcolor.rgb2hsv(rgb01)
    elif color_space == "LAB":
        out = skcolor.rgb2lab(rgb01)
    elif color_space == "XYZ":
        out = skcolor.rgb2xyz(rgb01)
    else:
        raise ValueError(f"Unknown color space: {color_space}")

    lo, hi = COLOR_SPACE_RANGES[color_space]
    out = (out - lo) / (hi - lo)      # -> [0,1] per channel
    out = np.clip(out, 0.0, 1.0)
    out = out * 2.0 - 1.0             # -> [-1,1]
    return out.astype(np.float32)

## 5. Dataset Class

In [ ]:
class ColorSpaceRegressionDataset(Dataset):
    '''Loads a 16-bit TIFF, converts to the requested color space, resizes to IMG_SIZE,
    optionally augments (train only), and returns (tensor[C,H,W], target[1]).'''

    def __init__(self, data_list, color_space="RGB", img_size=IMG_SIZE,
                 augment=False, target_noise_std=0.0):
        self.data_list = data_list
        self.color_space = color_space
        self.img_size = img_size
        self.augment = augment
        self.target_noise_std = target_noise_std
        self.resize = transforms.Resize((img_size, img_size), antialias=True)

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        path, target = self.data_list[idx]
        try:
            img = io.imread(path)
            if img.dtype != np.uint16:
                raise ValueError(f"{os.path.basename(path)} is not 16-bit (dtype={img.dtype})")
            img = img.astype(np.float32) / 65535.0            # -> [0,1] RGB
            img = convert_and_normalize(img, self.color_space)  # -> [-1,1] target color space
            img_t = torch.from_numpy(img.transpose(2, 0, 1)).float()
            img_t = self.resize(img_t)

            if self.augment:
                if random.random() < 0.5:
                    img_t = torch.flip(img_t, dims=[2])
                if random.random() < 0.5:
                    img_t = torch.flip(img_t, dims=[1])
                k = random.choice([0, 1, 2, 3])
                img_t = torch.rot90(img_t, k=k, dims=[1, 2])

            target_t = torch.tensor([target], dtype=torch.float32)
            if self.target_noise_std > 0:
                target_t = target_t + torch.randn_like(target_t) * self.target_noise_std
            return img_t, target_t

        except Exception as e:
            print(f"Error loading {path}: {e}")
            return torch.zeros(3, self.img_size, self.img_size), torch.tensor([target], dtype=torch.float32)

## 6. Model — `RegressionCNN`

Reused **verbatim** from `zucchini_cnn_regression_128.ipynb` (6 conv blocks, depth-doubling, ELU output, single regression head) so results stay comparable to that baseline. Not touched by this ablation.

In [ ]:
class RegressionCNN(nn.Module):
    '''Custom 2D CNN for regression -- identical to the base notebook's architecture.'''

    def __init__(self, input_channels=3, img_height=128, img_width=128, depth=16):
        super(RegressionCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(input_channels, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                    # 128 -> 64

            nn.Conv2d(depth, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.Conv2d(depth * 2, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                    # 64 -> 32

            nn.Conv2d(depth * 2, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.Conv2d(depth * 4, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                    # 32 -> 16

            nn.Conv2d(depth * 4, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.Conv2d(depth * 8, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                    # 16 -> 8

            nn.Conv2d(depth * 8, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.Conv2d(depth * 16, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                    # 8 -> 4

            nn.Conv2d(depth * 16, depth * 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 32), nn.ReLU(),
            nn.MaxPool2d(2, 2),                                    # 4 -> 2
        )

        self.flattened_size = depth * img_height * img_width // 128

        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size // 2),
            nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4),
            nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1),
            nn.ELU(),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

## 7. Training & Evaluation Utilities

Adds `Acc@0` (rounded prediction == true score) and `Acc@1` (within ±1) alongside MSE/RMSE/MAE/R², consistent with the metrics used across the other ablation notebooks in this project.

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "acc0": float(np.mean(np.round(y_pred) == np.round(y_true))),
        "acc1": float(np.mean(np.abs(np.round(y_pred) - y_true) <= 1.0)),
    }


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    val_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_targets, all_preds)
    return val_loss, metrics, np.array(all_preds).flatten(), np.array(all_targets).flatten()


def train_one_fold(train_list, val_list, color_space, epochs, early_stop_patience,
                    img_size=IMG_SIZE, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
                    lr_sched_patience=LR_SCHED_PATIENCE, target_noise_std=TARGET_NOISE_STD,
                    quiet=True):
    '''Trains a fresh RegressionCNN on one fold, returns (best_val_metrics, best_preds, best_targets).'''
    train_ds = ColorSpaceRegressionDataset(train_list, color_space, img_size,
                                            augment=True, target_noise_std=target_noise_std)
    val_ds   = ColorSpaceRegressionDataset(val_list, color_space, img_size,
                                            augment=False, target_noise_std=0.0)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = RegressionCNN(input_channels=3, img_height=img_size, img_width=img_size, depth=16).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1,
                                                       patience=lr_sched_patience)

    best_val_loss = float("inf")
    best_metrics, best_preds, best_targets = None, None, None
    best_state = None
    epochs_without_improvement = 0

    iterator = range(epochs) if quiet else tqdm(range(epochs), desc="epochs")
    for epoch in iterator:
        train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, metrics, preds, targets = validate_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_metrics = metrics
            best_preds, best_targets = preds, targets
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_patience:
            break

    return best_metrics, best_preds, best_targets, best_state

## 8. Cross-Validation Runner

Stratified K-fold on the integer score label. Each fold trains a **fresh** `RegressionCNN` from scratch (no weight sharing across folds) and keeps the best-val epoch's metrics, matching the CV protocol used elsewhere in this project.

In [ ]:
def run_cv_for_combo(filter_condition, color_space, data_list, k_folds, epochs, early_stop_patience,
                      seed=SEED, verbose=True):
    labels = [int(t) for _, t in data_list]
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)
    indices = np.arange(len(data_list))

    fold_records = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        train_list = [data_list[i] for i in train_idx]
        val_list   = [data_list[i] for i in val_idx]

        metrics, preds, targets, _ = train_one_fold(
            train_list, val_list, color_space, epochs=epochs,
            early_stop_patience=early_stop_patience, quiet=True,
        )
        metrics["fold"] = fold_i
        fold_records.append(metrics)
        if verbose:
            print(f"    fold {fold_i+1}/{k_folds}: R²={metrics['r2']:.4f}  RMSE={metrics['rmse']:.4f}  "
                  f"MAE={metrics['mae']:.4f}  Acc@1={metrics['acc1']:.3f}  (n_train={len(train_list)}, n_val={len(val_list)})")

    fold_df = pd.DataFrame(fold_records)
    agg = {}
    for m in ["mse", "rmse", "mae", "r2", "acc0", "acc1"]:
        agg[f"{m}_mean"] = fold_df[m].mean()
        agg[f"{m}_std"] = fold_df[m].std()
    agg["n_images"] = len(data_list)
    agg["n_folds"] = k_folds
    return agg, fold_df

## 9. Master Ablation Loop

Loops over all `(filter_condition, color_space)` combinations. **Checkpoints results to CSV after every combo** and skips combos already present on re-run, so a crash or a Restart Kernel partway through an overnight run doesn't lose completed work (the base notebook's own kernel-crash history is exactly why this matters here).

In [ ]:
results_csv_path = "ablation_cc_nocc_colorspace_results.csv"
fold_results_csv_path = "ablation_cc_nocc_colorspace_fold_results.csv"

results_records = []
fold_records_all = []
completed = set()
if os.path.exists(results_csv_path):
    prev = pd.read_csv(results_csv_path)
    completed = set(zip(prev["filter_condition"], prev["color_space"]))
    results_records = prev.to_dict("records")
    print(f"Resuming: {len(completed)} combo(s) already completed, will skip them.")
if os.path.exists(fold_results_csv_path):
    fold_records_all = pd.read_csv(fold_results_csv_path).to_dict("records")

for cond in FILTER_CONDITIONS:
    data_list = discovered.get(cond, [])
    if len(data_list) == 0:
        print(f"Skipping filter='{cond}' -- no images discovered in §3.")
        continue
    for cs in COLOR_SPACES:
        if (cond, cs) in completed:
            print(f"Skipping {cond}/{cs} -- already in {results_csv_path}.")
            continue

        print("\n" + "=" * 80)
        print(f"ABLATION RUN -- filter={cond} | color_space={cs} | n_images={len(data_list)}")
        print("=" * 80)

        agg, fold_df = run_cv_for_combo(
            cond, cs, data_list,
            k_folds=ABLATION_K_FOLDS, epochs=ABLATION_EPOCHS,
            early_stop_patience=ABLATION_EARLY_STOP_PATIENCE,
        )
        record = {"filter_condition": cond, "color_space": cs, **agg}
        results_records.append(record)

        fold_df = fold_df.copy()
        fold_df["filter_condition"] = cond
        fold_df["color_space"] = cs
        fold_records_all.extend(fold_df.to_dict("records"))

        pd.DataFrame(results_records).to_csv(results_csv_path, index=False)
        pd.DataFrame(fold_records_all).to_csv(fold_results_csv_path, index=False)

        print(f"  → R²={agg['r2_mean']:.4f}±{agg['r2_std']:.4f}  RMSE={agg['rmse_mean']:.4f}±{agg['rmse_std']:.4f}  "
              f"MAE={agg['mae_mean']:.4f}  Acc@1={agg['acc1_mean']:.3f}")

results_df = pd.DataFrame(results_records)
results_df.sort_values("r2_mean", ascending=False)

## 10. Ablation Results — Visualization

In [ ]:
# Heatmap of mean R² by filter x color space
pivot_r2 = results_df.pivot(index="filter_condition", columns="color_space", values="r2_mean")
pivot_r2 = pivot_r2.reindex(index=FILTER_CONDITIONS, columns=COLOR_SPACES)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot_r2.values, cmap="YlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot_r2.columns))); ax.set_xticklabels(pivot_r2.columns)
ax.set_yticks(range(len(pivot_r2.index))); ax.set_yticklabels(pivot_r2.index)
for i in range(pivot_r2.shape[0]):
    for j in range(pivot_r2.shape[1]):
        val = pivot_r2.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    color="black" if val < 0.75 else "white", fontweight="bold")
ax.set_title("Mean R² (5-fold CV) by filter × color space", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, label="R²")
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart with std error bars, all 6 metrics of interest
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_to_plot = [("r2_mean", "r2_std", "R²"), ("rmse_mean", "rmse_std", "RMSE"), ("acc1_mean", "acc1_std", "Acc@1")]
width = 0.35
x = np.arange(len(COLOR_SPACES))

for ax, (mean_col, std_col, label) in zip(axes, metrics_to_plot):
    for i, cond in enumerate(FILTER_CONDITIONS):
        sub = results_df[results_df["filter_condition"] == cond].set_index("color_space").reindex(COLOR_SPACES)
        ax.bar(x + (i - 0.5) * width, sub[mean_col], width, yerr=sub[std_col],
               capsize=4, label=cond, edgecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(COLOR_SPACES)
    ax.set_ylabel(label, fontsize=11, fontweight="bold")
    ax.set_title(f"{label} by color space", fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Ranked summary table
ranked = results_df.sort_values("r2_mean", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1
display_cols = ["rank", "filter_condition", "color_space", "r2_mean", "r2_std",
                 "rmse_mean", "mae_mean", "acc0_mean", "acc1_mean", "n_images"]
print("=" * 80)
print("RANKED ABLATION RESULTS (by mean R²)")
print("=" * 80)
print(ranked[display_cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

best_row = ranked.iloc[0]
BEST_FILTER, BEST_COLOR_SPACE = best_row["filter_condition"], best_row["color_space"]
print(f"\n★ Best combination: filter='{BEST_FILTER}', color_space='{BEST_COLOR_SPACE}' "
      f"(R²={best_row['r2_mean']:.4f}±{best_row['r2_std']:.4f})")

## 11. Winning-Combo Deep Dive

Retrains the best `(filter, color_space)` combination on a single stratified 80/20 split at **full scale** (`FINAL_EPOCHS` / `FINAL_EARLY_STOP_PATIENCE`, unaffected by `QUICK_MODE`), then produces the scatter/residual/box plots and prediction "frieze" in the same style as the base notebook. Set `BEST_FILTER` / `BEST_COLOR_SPACE` manually here instead of the previous cell's output if you want to inspect a different combo.

In [ ]:
print(f"Deep dive on: filter='{BEST_FILTER}', color_space='{BEST_COLOR_SPACE}'")
deep_dive_data = discovered[BEST_FILTER]
deep_dive_labels = [int(t) for _, t in deep_dive_data]

train_list, val_list = train_test_split(
    deep_dive_data, test_size=0.2, stratify=deep_dive_labels, random_state=SEED
)
print(f"  train={len(train_list)}  val={len(val_list)}")

final_metrics, final_preds, final_targets, final_state = train_one_fold(
    train_list, val_list, BEST_COLOR_SPACE,
    epochs=FINAL_EPOCHS, early_stop_patience=FINAL_EARLY_STOP_PATIENCE, quiet=False,
)

print("\n" + "=" * 80)
print(f"FINAL MODEL PERFORMANCE  (filter={BEST_FILTER}, color_space={BEST_COLOR_SPACE})")
print("=" * 80)
for k, v in final_metrics.items():
    print(f"{k:>6}: {v:.4f}")

final_model_path = f"best_regression_cnn_{BEST_FILTER}_{BEST_COLOR_SPACE}.pth"
torch.save(final_state, final_model_path)
print(f"\nSaved weights to: {final_model_path}")

In [ ]:
# Scatter (predicted vs true) + residuals
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(final_targets, final_preds, alpha=0.6, s=50, edgecolors="black", linewidth=0.5)
ax.plot([final_targets.min(), final_targets.max()], [final_targets.min(), final_targets.max()],
        "r--", linewidth=2, label="Perfect Prediction")
ax.set_xlabel("True Value", fontweight="bold"); ax.set_ylabel("Predicted Value", fontweight="bold")
ax.set_title(f"Predicted vs True (R²={final_metrics['r2']:.3f})", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
residuals = final_preds - final_targets
ax.scatter(final_targets, residuals, alpha=0.6, s=50, edgecolors="black", linewidth=0.5)
ax.axhline(y=0, color="r", linestyle="--", linewidth=2)
ax.set_xlabel("True Value", fontweight="bold"); ax.set_ylabel("Residual (Pred - True)", fontweight="bold")
ax.set_title(f"Residuals (MAE={final_metrics['mae']:.3f})", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box plot of predictions per true class (winning combo)
fig, ax = plt.subplots(figsize=(14, 6))
unique_targets = np.unique(final_targets)
data_to_plot = [final_preds[final_targets == t] for t in unique_targets]

bp = ax.boxplot(data_to_plot, positions=unique_targets, widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor="lightblue", edgecolor="black", linewidth=1.5),
                 medianprops=dict(color="red", linewidth=2))
for t in unique_targets:
    ax.axhline(y=t, color="green", linestyle="--", alpha=0.5, linewidth=1)

ax.set_xlabel("True Class (Score)", fontweight="bold")
ax.set_ylabel("Predicted Value", fontweight="bold")
ax.set_title(f"Prediction distribution by true class -- {BEST_FILTER}/{BEST_COLOR_SPACE}", fontweight="bold")
ax.set_xticks(unique_targets)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### Prediction frieze

Reloads each validation image's **original RGB** rendering (independent of whichever color space fed the network) purely for display, sorted by predicted score, with a class-boundary marker whenever the rounded prediction crosses into the next integer score.

In [ ]:
def load_display_rgb(path, img_size=IMG_SIZE):
    '''Reload the raw TIFF as plain [0,1] RGB, resized, for visualization only.'''
    img = io.imread(path).astype(np.float32) / 65535.0
    img_t = torch.from_numpy(img.transpose(2, 0, 1)).float()
    img_t = transforms.Resize((img_size, img_size), antialias=True)(img_t)
    return np.clip(img_t.permute(1, 2, 0).numpy(), 0, 1)

val_paths = [p for p, _ in val_list]

sorted_idx = np.argsort(final_preds)
sorted_paths = [val_paths[i] for i in sorted_idx]
sorted_preds = final_preds[sorted_idx]

fig, ax = plt.subplots(1, len(sorted_paths) + 9, figsize=(len(sorted_paths) + 9, 1))
class_int = 0
for i, (path, pred) in enumerate(zip(sorted_paths, sorted_preds)):
    img = load_display_rgb(path)
    class_int_tmp = int(np.floor(pred))
    if class_int_tmp > class_int:
        ax[i + class_int].axis("off")
        ax[i + class_int].text(0.5, 0.5, f"{class_int_tmp}", fontsize=30, fontweight="bold",
                                ha="center", va="center")
        while class_int < class_int_tmp:
            class_int += 1
            ax[i + class_int].set_title(f"{pred:.2f}", fontsize=12)
            ax[i + class_int].imshow(img)
            ax[i + class_int].axis("off")
    else:
        ax[i + class_int].set_title(f"{pred:.2f}", fontsize=12)
        ax[i + class_int].imshow(img)
        ax[i + class_int].axis("off")
while class_int < 9:
    class_int += 1
    ax[len(sorted_paths) + class_int - 1].axis("off")
    ax[len(sorted_paths) + class_int - 1].text(0.5, 0.5, f"{class_int}", fontsize=30,
                                                fontweight="bold", ha="center", va="center")
plt.show()

## 12. Export

Everything is already checkpointed incrementally during §9 (`ablation_cc_nocc_colorspace_results.csv`, `ablation_cc_nocc_colorspace_fold_results.csv`) and the winning model is saved in §11. This cell just prints a final summary of what was written.

In [ ]:
print("Files written to the current working directory:")
for f in [results_csv_path, fold_results_csv_path, final_model_path]:
    if os.path.exists(f):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f}  ({size_kb:.1f} KB)")
    else:
        print(f"  {f}  (NOT FOUND)")

print("\nSummary:")
print(f"  Combinations evaluated: {len(results_df)} / {len(FILTER_CONDITIONS) * len(COLOR_SPACES)}")
print(f"  Best combo: filter='{BEST_FILTER}', color_space='{BEST_COLOR_SPACE}', "
      f"R²={best_row['r2_mean']:.4f}±{best_row['r2_std']:.4f}")